# Exportação do Modelo para o Aplicativo Flutter

Gera os arquivos que o app (`app/`) carrega de `assets/modelo/`:

| Arquivo | Conteúdo |
|---|---|
| `modelo.tflite` | Modelo escolhido (padrão: **EfficientNetV2B0 × BRACOL**) com **duas saídas**: os *logits* e os mapas **Grad-CAM de todas as classes** |
| `config.json` | Classes (rótulo + nome em português), temperatura da calibração, limiar de abstenção e métricas do modelo exportado no teste |

**Por que o Grad-CAM vai dentro do `.tflite`:** o TFLite não calcula gradientes. Porém, entre o mapa de
características e a saída só há `GlobalAveragePooling → [Dense+ReLU → BatchNorm → Dropout] × N → Dense+softmax`.
Essa cabeça é pequena, então a sua derivada pode ser escrita **explicitamente** com operações comuns
(multiplicação de matrizes, máscara do ReLU, escala do BatchNorm). O resultado é **o mesmo Grad-CAM** de
`make_gradcam_heatmap` do `mobilenet.ipynb` (gradiente da probabilidade softmax da classe em relação ao mapa
de características do backbone). A célula de verificação compara os dois numericamente.

**Pré-requisito:** o arquivo `.keras` do modelo treinado (gerado pelo `mobilenet.ipynb` em
`outputs/{Modelo}_{dataset}/{Modelo}_{dataset}.keras`). Anexe-o como *Input* (Dataset ou Output de uma versão
salva do notebook das CNNs) ou coloque-o em `outputs/` desta sessão.

In [ ]:
import os
import glob
import json
import time
import shutil
import zipfile
import re
import unicodedata

import numpy as np
import tensorflow as tf
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, f1_score

print(f"TensorFlow: {tf.__version__}")

# ── Modelo a exportar ─────────────────────────────────────────────────────────
# EfficientNetV2B0 × BRACOL: melhor F1-Macro da Avaliação Cruzada nos 3 datasets comparáveis, 7 MB em TFLite
# e ~37 ms por imagem em CPU. BRACOL é o único com as 5 classes (Ferrugem, Bicho-Mineiro, Cercosporiose,
# Phoma e Saudável). Funciona com qualquer combinação treinada pelo mobilenet.ipynb.
MODELO_APP  = 'EfficientNetV2B0'
DATASET_APP = 'bracol'
NOME_ARQUIVO = f'{MODELO_APP}_{DATASET_APP}'

# Mesmos caminhos do mobilenet.ipynb (val: ajuste da temperatura; test: verificação do modelo exportado)
DATASETS = {
    'bracol': {
        'train': '/kaggle/input/newdataset/new_dataset/train',
        'val':   '/kaggle/input/newdataset/new_dataset/val',
        'test':  '/kaggle/input/newdataset/new_dataset/test',
    },
    'rocole': {
        'train': '/kaggle/working/datasets/rocole/train',
        'val':   '/kaggle/working/datasets/rocole/val',
        'test':  '/kaggle/working/datasets/rocole/test',
    },
    'coffee_leaf_diseases': {
        'train': '/kaggle/working/datasets/coffee_leaf_diseases/train',
        'val':   '/kaggle/working/datasets/coffee_leaf_diseases/val',
        'test':  '/kaggle/working/datasets/coffee_leaf_diseases/test',
    },
    'jmuben': {
        'train': '/kaggle/working/datasets/jmuben/train',
        'val':   '/kaggle/working/datasets/jmuben/val',
        'test':  '/kaggle/working/datasets/jmuben/test',
    },
}

IMG_SIZE = 224
LIMIAR_ABSTENCAO = 0.70   # confiança calibrada mínima para o app mostrar o diagnóstico sem aviso (ver tabela de abstenção do mobilenet.ipynb)
SAIDA_APP = 'app_assets'
os.makedirs(SAIDA_APP, exist_ok=True)

# ── Localiza o .keras ─────────────────────────────────────────────────────────
candidatos = sorted(
    glob.glob(os.path.join('outputs', NOME_ARQUIVO, f'{NOME_ARQUIVO}.keras'))
    + glob.glob(f'/kaggle/input/**/{NOME_ARQUIVO}.keras', recursive=True)
)
if not candidatos:
    raise FileNotFoundError(
        f"Não encontrei {NOME_ARQUIVO}.keras em outputs/ nem em /kaggle/input. Anexe o Output do mobilenet.ipynb "
        f"(Add Input → Notebooks) ou rode-o com BACKBONE_ATIVO='{MODELO_APP}' e DATASET_ATIVO='{DATASET_APP}'."
    )
CAMINHO_KERAS = candidatos[0]
print(f"Modelo: {CAMINHO_KERAS}")

model = tf.keras.models.load_model(CAMINHO_KERAS, compile=False)
print(f"Entrada: {model.input_shape} | Saída: {model.output_shape}")

# Classes e nomes exibidos no app

In [ ]:
def _n_classes(pasta):
    return sum(os.path.isdir(os.path.join(pasta, d)) for d in os.listdir(pasta))


def _resolver_splits(cfg, n_classes):
    """Usa os caminhos de DATASETS se existirem; senão procura em /kaggle/input uma pasta com train/, val/ e
    test/ cujo train/ tenha o mesmo número de classes do modelo (o caminho de montagem muda conforme o nome
    com que o dataset foi anexado, ex.: /kaggle/input/bracol/... ou /kaggle/input/datasets/<dono>/<nome>/...)."""
    if all(os.path.isdir(cfg[s]) for s in ('train', 'val', 'test')):
        return cfg
    for train in sorted(glob.glob('/kaggle/input/**/train', recursive=True)):
        base = os.path.dirname(train)
        if all(os.path.isdir(os.path.join(base, s)) for s in ('val', 'test')) and _n_classes(train) == n_classes:
            print(f"Caminho de DATASETS['{DATASET_APP}'] não existe -- usando {base}")
            return {s: os.path.join(base, s) for s in ('train', 'val', 'test')}
    raise FileNotFoundError(
        f"Não encontrei train/val/test com {n_classes} classes em /kaggle/input. Rode "
        "`!find /kaggle/input -maxdepth 5 -type d -name train` e ajuste DATASETS na primeira célula."
    )


# Mesma ordem que o image_dataset_from_directory usou no treino: subpastas do treino em ordem alfabética.
cfg_ds = _resolver_splits(DATASETS[DATASET_APP], model.output_shape[-1])
print(f"Dataset: {os.path.dirname(cfg_ds['train'])}")
rotulos = sorted(d for d in os.listdir(cfg_ds['train']) if os.path.isdir(os.path.join(cfg_ds['train'], d)))
assert len(rotulos) == model.output_shape[-1], f"{len(rotulos)} pastas de classe, mas o modelo tem {model.output_shape[-1]} saídas"


def _normalizar(nome):
    sem_acento = unicodedata.normalize('NFKD', nome).encode('ascii', 'ignore').decode()
    nome = sem_acento.strip().lower().replace('-', '_').replace(' ', '_')
    return re.sub(r'^\d+_', '', nome)   # BRACOL numera as pastas: "1_health", "5_cercospora"...


# Nome amigável por rótulo de pasta (cobre os vocabulários em inglês e português dos 4 datasets).
# Rótulos fora desta lista aparecem no app com a primeira letra maiúscula -- edite config.json se quiser.
NOMES_PT = {
    'ferrugem': 'Ferrugem', 'rust': 'Ferrugem', 'leaf_rust': 'Ferrugem', 'coffee_rust': 'Ferrugem',
    'bicho_mineiro': 'Bicho-mineiro', 'miner': 'Bicho-mineiro', 'leaf_miner': 'Bicho-mineiro',
    'cercosporiose': 'Cercosporiose', 'cercospora': 'Cercosporiose', 'cerscospora': 'Cercosporiose',
    'brown_eye_spot': 'Cercosporiose',
    'phoma': 'Phoma',
    'saudavel': 'Saudável', 'healthy': 'Saudável', 'health': 'Saudável', 'sadia': 'Saudável', 'nodisease': 'Saudável',
    'acaro_vermelho': 'Ácaro-vermelho', 'red_spider_mite': 'Ácaro-vermelho', 'red_mite': 'Ácaro-vermelho',
}
classes_app = [{'rotulo': r, 'nome': NOMES_PT.get(_normalizar(r), r.replace('_', ' ').capitalize())} for r in rotulos]
for c in classes_app:
    print(f"  {c['rotulo']:20s} -> {c['nome']}")

# Modelo explicável: logits + Grad-CAM numa única inferência

In [ ]:
# Extrator: imagem (0-255) -> mapa de características do backbone, no grafo EXTERNO do modelo.
# Usar a ENTRADA da camada 'global_avg_pool' (em vez da saída de 'base_model', que é um modelo aninhado) garante
# que o pré-processamento do backbone continua incluído e que o grafo fica conectado em qualquer versão do Keras.
extrator = tf.keras.Model(model.inputs, model.get_layer('global_avg_pool').input, name='extrator')

n_blocos = len([l for l in model.layers if l.name.startswith('dense_features_l')])
blocos = []
for i in range(n_blocos):
    W, b = model.get_layer(f'dense_features_l{i}').get_weights()
    bn = model.get_layer(f'bn_l{i}')
    gamma, beta, media_movel, var_movel = bn.get_weights()
    escala = gamma / np.sqrt(var_movel + bn.epsilon)
    deslocamento = beta - media_movel * escala
    # Dropout é identidade na inferência -- não entra.
    blocos.append(tuple(tf.constant(v, tf.float32) for v in (W, b, escala, deslocamento)))
W_final, b_final = (tf.constant(v, tf.float32) for v in model.get_layer('predicao_final').get_weights())
N_CLASSES = int(W_final.shape[1])
print(f"Cabeça: {n_blocos} bloco(s) denso(s) + saída com {N_CLASSES} classes")


def _cabeca(mapa):
    """GAP + blocos densos + logits. Devolve também o que a retropropagação manual precisa."""
    x = tf.reduce_mean(mapa, axis=[1, 2])
    cache = []
    for W, b, escala, deslocamento in blocos:
        z = tf.matmul(x, W) + b
        x = tf.nn.relu(z) * escala + deslocamento      # Dense(ReLU) -> BatchNorm (inferência)
        cache.append((W, z, escala))
    return tf.matmul(x, W_final) + b_final, cache


class GradCamExplicito(tf.keras.layers.Layer):
    """Recebe o mapa de características do backbone e devolve (logits, Grad-CAM de todas as classes)."""

    def call(self, mapa):
        mapa = tf.cast(mapa, tf.float32)                                    # (1, h, w, C)
        h, w, C = mapa.shape[1], mapa.shape[2], mapa.shape[3]
        logits, cache = _cabeca(mapa)
        p = tf.nn.softmax(logits)                                           # (1, n)

        # d p_c / d x_final = p_c * sum_j p_j * (W_final[:, c] - W_final[:, j])   (derivada da softmax, todas as
        # classes de uma vez). Escrita com as DIFERENÇAS W_c - W_j em vez de p_c * (W_c - W_final @ p): as duas são
        # iguais na matemática, mas com o modelo confiante (p_c ~ 0,9999) a segunda subtrai dois vetores quase
        # idênticos e perde toda a precisão em float32 -- o próprio GradientTape sofre disso nesse regime.
        W_t = tf.transpose(W_final)                                         # (n, H)
        difs = tf.expand_dims(W_t, 1) - tf.expand_dims(W_t, 0)              # (n, n, H): W_c - W_j
        G = tf.transpose(p) * tf.reduce_sum(difs * tf.reshape(p, [1, N_CLASSES, 1]), axis=1)   # (n, H)
        for W, z, escala in reversed(cache):
            G = G * escala                                                  # BatchNorm
            G = G * tf.cast(z > 0, tf.float32)                              # ReLU
            G = tf.matmul(G, W, transpose_b=True)                           # Dense
        # G = d p_c / d GAP(mapa). Pelo GAP, d p_c / d mapa[i,j,k] = G[c,k] / (h*w) em toda posição, ou seja,
        # G é (a menos da constante 1/(h*w), que some na normalização) exatamente o "pooled_grads" do Grad-CAM.
        cams = tf.matmul(G, tf.reshape(mapa, [h * w, C]), transpose_b=True)  # (n, h*w)
        cams = tf.reshape(tf.nn.relu(cams), [N_CLASSES, h, w])
        return logits, cams


# Modelo Keras comum (entrada com lote fixo 1, como no app) -> exportado com from_keras_model, o mesmo caminho
# já usado para o .tflite no mobilenet.ipynb. (from_concrete_functions/SavedModel de um tf.Module quebra no
# conversor MLIR com Keras 3: "missing attribute 'value'" nas BatchNorms do backbone.)
entrada_app = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), batch_size=1, name='imagem')
# training=False EXPLÍCITO: o create_model chama o backbone com training=fine_t, e com fine_tune=True o Keras grava
# training=True no grafo. model(img, training=False) sobrescreve isso, mas um modelo que chama o extrator sem
# training herdaria o True -- as BatchNorms do backbone normalizariam cada foto com as estatísticas dela mesma.
logits_app, cams_app = GradCamExplicito(name='gradcam_explicito')(extrator(entrada_app, training=False))
explicador = tf.keras.Model(entrada_app, {'logits': logits_app, 'gradcam': cams_app}, name='explicador_app')

saida_teste = explicador(tf.zeros([1, IMG_SIZE, IMG_SIZE, 3]))
GRADE_CAM = list(saida_teste['gradcam'].shape[1:])
print(f"Saídas: logits {saida_teste['logits'].shape} | gradcam {saida_teste['gradcam'].shape}")

# Verificação: mesmas probabilidades e mesmo Grad-CAM do notebook

In [ ]:
def gradcam_referencia(img_array, pred_index):
    """Grad-CAM pelo GradientTape -- mesma definição de make_gradcam_heatmap do mobilenet.ipynb."""
    idx_gap = model.layers.index(model.get_layer('global_avg_pool'))
    with tf.GradientTape() as tape:
        mapa = extrator(img_array, training=False)
        tape.watch(mapa)
        out = mapa
        for layer in model.layers[idx_gap:]:
            out = layer(out, training=False)
        classe = out[:, pred_index]
    grads = tape.gradient(classe, mapa)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heat = tf.squeeze(mapa[0] @ pooled[..., tf.newaxis])
    return (tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-12)).numpy()


# Cabeça em NumPy float64 (implementação independente da camada acima) para conferir as probabilidades e, por
# diferenças finitas, o gradiente -- referência exata mesmo com a softmax saturada.
_blocos64 = [tuple(t.numpy().astype(np.float64) for t in bl) for bl in blocos]
_Wf64, _bf64 = W_final.numpy().astype(np.float64), b_final.numpy().astype(np.float64)


def _probs64(g):
    x = g
    for W, b, escala, deslocamento in _blocos64:
        x = np.maximum(x @ W + b, 0) * escala + deslocamento
    z = x @ _Wf64 + _bf64
    z = z - z.max(axis=-1, keepdims=True)
    return np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)


# eps pequeno (1e-5): passos maiores atravessam as "quinas" do ReLU da cabeça e a diferença finita deixa de ser
# uma boa aproximação da derivada (com 1e-3 o erro chega a ~6%; com 1e-5 fica em ~1e-9).
def gradcam_numerico(img_array, classe, eps=1e-5):
    mapa = extrator(img_array, training=False).numpy().astype(np.float64)[0]   # (h, w, C)
    g = mapa.mean(axis=(0, 1))
    pert = np.eye(len(g)) * eps
    # d p_c = -d(1 - p_c): o complemento (soma das OUTRAS classes) é pequeno e exato mesmo com p_c ~ 0,99999999,
    # enquanto p_c(g + eps) - p_c(g - eps) subtrairia dois números quase iguais a 1.
    resto = lambda x: np.delete(_probs64(x), classe, axis=-1).sum(axis=-1)
    grad = -(resto(g + pert) - resto(g - pert)) / (2 * eps)                  # d p_c / d GAP
    cam = np.maximum(mapa @ grad, 0)
    return cam, _probs64(g[None])[0]


ds_teste = tf.keras.utils.image_dataset_from_directory(
    cfg_ds['test'], image_size=(IMG_SIZE, IMG_SIZE), batch_size=1, label_mode='int', shuffle=False
)
dif_probs, dif_cams, dif_tape, conf = [], [], [], []
for img, _ in ds_teste.take(10):
    ref = model(img, training=False).numpy()[0]
    out = explicador(img)                       # sem training -- exatamente como o conversor TFLite chama
    nossa = tf.nn.softmax(out['logits']).numpy()[0]
    c = int(np.argmax(ref))
    cam_ref, probs64 = gradcam_numerico(img, c)
    dif_probs.append(max(np.abs(ref - nossa).max(), np.abs(probs64 - ref).max()))
    conf.append(ref[c])
    if cam_ref.max() > 0:
        cam = out['gradcam'].numpy()[c]
        dif_cams.append(np.abs(cam_ref / cam_ref.max() - cam / (cam.max() + 1e-30)).max())
        dif_tape.append(np.abs(cam_ref / cam_ref.max() - gradcam_referencia(img, c)).max())

print(f"Confiança média do modelo nessas imagens: {np.mean(conf):.6f}")
print(f"Maior diferença de probabilidade (modelo original × explicador): {max(dif_probs):.2e}")
print(f"Maior diferença no Grad-CAM normalizado (explicador × gradiente numérico float64): {max(dif_cams):.2e}")
print(f"  (informativo) GradientTape float32 do notebook × gradiente numérico: {max(dif_tape):.2e} "
      "-- pode ser grande com a softmax saturada; é imprecisão do float32, não do explicador")
assert max(dif_probs) < 1e-3, "Probabilidades do explicador divergem do modelo original -- não exporte."
assert max(dif_cams) < 2e-2, "Grad-CAM do explicador diverge do gradiente numérico -- não exporte."

# Calibração (Temperature Scaling) no conjunto de validação

In [ ]:
# Mesmo procedimento do mobilenet.ipynb: um escalar T ajustado na VALIDAÇÃO minimizando a NLL. O app usa
# softmax(logits / T) para exibir a confiança e decidir quando pedir outra foto.
@tf.function
def logits_lote(imgs):
    return _cabeca(tf.cast(extrator(imgs, training=False), tf.float32))[0]


ds_val = tf.keras.utils.image_dataset_from_directory(
    cfg_ds['val'], image_size=(IMG_SIZE, IMG_SIZE), batch_size=32, label_mode='int', shuffle=False
)
logits_val = np.concatenate([logits_lote(x).numpy() for x, _ in ds_val])
y_val = np.concatenate([y.numpy() for _, y in ds_val])


def nll(T):
    z = logits_val / T
    z = z - z.max(axis=1, keepdims=True)
    log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    return -log_p[np.arange(len(y_val)), y_val].mean()


TEMPERATURA = float(minimize_scalar(nll, bounds=(0.05, 10.0), method='bounded').x)
print(f"Temperatura ajustada: T = {TEMPERATURA:.4f}  (NLL {nll(1.0):.4f} -> {nll(TEMPERATURA):.4f})")

# Conversão para TFLite e verificação do arquivo exportado

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(explicador)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # mesma quantização dinâmica (pesos INT8) do mobilenet.ipynb
tflite_bytes = converter.convert()

caminho_tflite = os.path.join(SAIDA_APP, 'modelo.tflite')
with open(caminho_tflite, 'wb') as f:
    f.write(tflite_bytes)
print(f"modelo.tflite: {len(tflite_bytes) / 1024 ** 2:.2f} MB")

interp = tf.lite.Interpreter(model_path=caminho_tflite)
interp.allocate_tensors()
entrada = interp.get_input_details()[0]
saidas = interp.get_output_details()
idx_logits = next(s['index'] for s in saidas if len(s['shape']) == 2)
idx_cam = next(s['index'] for s in saidas if len(s['shape']) == 3)
print(f"Entrada {entrada['shape']} {entrada['dtype'].__name__} | Saídas: "
      + ', '.join(f"{s['name']} {list(s['shape'])}" for s in saidas))

# Avalia o .tflite QUANTIZADO no teste inteiro -- são estas as métricas que o app vai de fato entregar
y_true, y_pred, conc, corr, tempos = [], [], [], [], []
for img, y in ds_teste:
    t0 = time.perf_counter()
    interp.set_tensor(entrada['index'], img.numpy().astype(np.float32))
    interp.invoke()
    tempos.append((time.perf_counter() - t0) * 1000)
    logits = interp.get_tensor(idx_logits)[0]
    pred = int(np.argmax(logits))
    y_true.append(int(y.numpy()[0]))
    y_pred.append(pred)
    if len(corr) < 20:   # Grad-CAM do quantizado × float32, nas primeiras 20 imagens
        cam_q = interp.get_tensor(idx_cam)[pred].ravel()
        cam_f = explicador(img)['gradcam'].numpy()[pred].ravel()
        if cam_q.std() > 0 and cam_f.std() > 0:
            corr.append(np.corrcoef(cam_q, cam_f)[0, 1])
    conc.append(pred == int(np.argmax(model(img, training=False).numpy()[0])))

metricas_teste = {
    'acuracia':        round(float(accuracy_score(y_true, y_pred)), 4),
    'f1_macro':        round(float(f1_score(y_true, y_pred, average='macro')), 4),
    'n_imagens':       len(y_true),
    'latencia_ms_cpu': round(float(np.mean(tempos)), 1),
}
print(f"Teste (TFLite quantizado): {metricas_teste}")
print(f"Concordância com o modelo Keras: {np.mean(conc) * 100:.2f}%")
print(f"Correlação média do Grad-CAM quantizado × float32: {np.mean(corr):.4f}")

# Arquivos para o app

In [ ]:
config_app = {
    'versao':            1,
    'modelo':            MODELO_APP,
    'dataset':           DATASET_APP,
    'tamanho_entrada':   IMG_SIZE,
    'grade_gradcam':     GRADE_CAM,
    'temperatura':       round(TEMPERATURA, 6),
    'limiar_abstencao':  LIMIAR_ABSTENCAO,
    'classes':           classes_app,
    'metricas_teste':    metricas_teste,
}
with open(os.path.join(SAIDA_APP, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(config_app, f, ensure_ascii=False, indent=2)
print(json.dumps(config_app, ensure_ascii=False, indent=2))

caminho_zip = f'{SAIDA_APP}.zip'
with zipfile.ZipFile(caminho_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for nome in ('modelo.tflite', 'config.json'):
        z.write(os.path.join(SAIDA_APP, nome), nome)
print(f"\nBaixe {caminho_zip} (painel Output → ⋮ → Download) e extraia em app/assets/modelo/.")